## Notebook 9 – Grouping & Aggregation

### Setup
For this notebook I only need pandas and NumPy. The command below installs them. If they are already installed, pip just says "Requirement already satisfied", so it is safe to run again. I used %pip so that the libraries are installed in the same environment as this notebook.

In [1]:
%pip install pandas numpy

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: C:\Users\rafiy\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import pandas as pd

For this notebook I use the sales table of an electronics shop. It has 12 orders with the columns order_id, region, product, quarter, units (number of items sold) and revenue (in thousands of rupees). The three products have a fixed price per unit (Laptop 50, Phone 20 and Tablet 15), so the numbers are easy to check by hand.

In [3]:
sales = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12],
    "region": ["North", "South", "East", "North", "South", "East",
               "North", "South", "East", "North", "South", "East"],
    "product": ["Laptop", "Phone", "Laptop", "Phone", "Tablet", "Phone",
                "Tablet", "Laptop", "Tablet", "Laptop", "Phone", "Phone"],
    "quarter": ["Q1", "Q1", "Q1", "Q1", "Q1", "Q2",
                "Q2", "Q2", "Q2", "Q2", "Q2", "Q1"],
    "units": [5, 10, 4, 8, 6, 12, 7, 3, 9, 6, 11, 5],
    "revenue": [250, 200, 200, 160, 90, 240, 105, 150, 135, 300, 220, 100]
})

print(sales)
print()
print("Shape:", sales.shape)

    order_id region product quarter  units  revenue
0          1  North  Laptop      Q1      5      250
1          2  South   Phone      Q1     10      200
2          3   East  Laptop      Q1      4      200
3          4  North   Phone      Q1      8      160
4          5  South  Tablet      Q1      6       90
5          6   East   Phone      Q2     12      240
6          7  North  Tablet      Q2      7      105
7          8  South  Laptop      Q2      3      150
8          9   East  Tablet      Q2      9      135
9         10  North  Laptop      Q2      6      300
10        11  South   Phone      Q2     11      220
11        12   East   Phone      Q1      5      100

Shape: (12, 6)


## Topic 1 – groupby()

### Step 1 – Understand the Concept
groupby() splits the table into groups according to the values of one or more columns, then a calculation is done on each group, and the results are combined into one table. This is called split-apply-combine.

The steps are:
1. Split : rows with the same value in the group column go into one group (for example all the North rows).
2. Apply : a calculation like sum(), mean(), count() or size() is done on every group.
3. Combine : the results of all the groups are joined into one Series or DataFrame.

Useful points:
- df.groupby("col")["other"].sum() gives one number per group. The group names become the index of the result.
- A list of columns, like ["region", "product"], makes groups from the combinations.
- as_index=False keeps the group column as a normal column and not as the index.
- size() counts the rows of each group, and count() counts the non-missing values of a column.
- get_group("name") gives the rows of one group.
- Always choose the columns to calculate (like ["revenue"]), because a calculation like sum() on text columns is not meaningful.

### Step 2 – Demonstrate the Concept

#### Business Example
The sales manager wants to know the total revenue of each region. The rows are grouped by the region column, the revenue of each group is added, and the result is one number for North, South and East.

### Step 3 – Implement the Concept

In [4]:
# Total revenue of each region
print(sales.groupby("region")["revenue"].sum())

# Number of groups, and number of rows in each group
print()
print("Number of groups:", sales.groupby("region").ngroups)
print(sales.groupby("region").size())

# Average units per region
print()
print(sales.groupby("region")["units"].mean())

region
East     675
North    815
South    660
Name: revenue, dtype: int64

Number of groups: 3
region
East     4
North    4
South    4
dtype: int64

region
East     7.5
North    6.5
South    7.5
Name: units, dtype: float64


In [5]:
# Many columns at once
print(sales.groupby("region")[["units", "revenue"]].sum())

# Group by two columns
print()
print(sales.groupby(["region", "product"])["revenue"].sum())

# Keeping the group column as a normal column
print()
print(sales.groupby("product", as_index=False)["revenue"].sum())

        units  revenue
region                
East       30      675
North      26      815
South      30      660

region  product
East    Laptop     200
        Phone      340
        Tablet     135
North   Laptop     550
        Phone      160
        Tablet     105
South   Laptop     150
        Phone      420
        Tablet      90
Name: revenue, dtype: int64

  product  revenue
0  Laptop      900
1   Phone      920
2  Tablet      330


In [6]:
# Looking at the groups one by one
for name, group in sales.groupby("quarter"):
    print(name, group.shape)

# The rows of a single group
print()
print(sales.groupby("region").get_group("North"))

Q1 (6, 6)
Q2 (6, 6)

   order_id region product quarter  units  revenue
0         1  North  Laptop      Q1      5      250
3         4  North   Phone      Q1      8      160
6         7  North  Tablet      Q2      7      105
9        10  North  Laptop      Q2      6      300


#### What does this program do?
In the first cell, the rows are grouped by region and the revenue is added in each group. The output is East 675, North 815 and South 660. The groups are sorted by name, so East comes first. There are 3 groups and each region has 4 rows. The average units per region is East 7.5, North 6.5 and South 7.5.

In the second cell, two columns are added together for each region (units and revenue), which gives East 30 units and 675, North 26 units and 815, and South 30 units and 660. Grouping by two columns, region and product, makes 9 groups, for example North with Laptop is 550 and South with Phone is 420. With as_index=False, the product groups come back as a normal table with the columns product and revenue: Laptop 900, Phone 920 and Tablet 330.

In the third cell, the for loop goes through the groups of the quarter column. Each group is a small table, and the output shows that Q1 and Q2 both have the shape (6, 6), which means 6 rows and 6 columns. get_group("North") gives the 4 rows of the North region.

#### AI/ML Example
groupby() is used to understand a dataset before modelling. For example, the average of a feature for each class (like the average income for each customer segment) shows which features can separate the classes. Groups are also used to create new features, such as the average purchase of each customer, and to check if the data is balanced in the groups.

#### Advantages
- Replaces long loops with one line of code.
- Works with one or many columns for grouping.
- Gives a summary of a big table very quickly.

#### Limitations
- The result of a group calculation has fewer rows than the original table, so the row level information is lost (transform() can keep it).
- Calculations on text columns can give meaningless results, so the columns must be chosen.
- Rows with a missing value in the group column are left out by default.

#### Key Point
groupby() splits the rows into groups, calculates for each group and combines the results. Choose the columns to calculate, like df.groupby("region")["revenue"].sum().

## Topic 2 – agg()

### Step 1 – Understand the Concept
agg() (short for aggregate) calculates more than one summary in the same step. It is used after groupby() but it also works on a normal Series or DataFrame.

Ways to use it:
- A list of functions: ["sum", "mean", "max"] gives one column for each function.
- A dictionary: {"units": "sum", "revenue": "mean"} uses a different function for each column.
- Named aggregation: new_name=("column", "function") gives the result columns the names we choose. This is the cleanest way.
- A custom function: any function that takes a Series and gives one value can be used.

### Step 2 – Demonstrate the Concept

#### Business Example
For every region the manager does not want only the total, but also the average and the highest order. Instead of three separate calculations, agg() gives all of them together in one table.

### Step 3 – Implement the Concept

In [7]:
# Many functions on one column
print(sales.groupby("region")["revenue"].agg(["sum", "mean", "max"]))

# A different function for each column
print()
print(sales.groupby("region").agg({"units": "sum", "revenue": "mean"}))

        sum    mean  max
region                  
East    675  168.75  240
North   815  203.75  300
South   660  165.00  220

        units  revenue
region                
East       30   168.75
North      26   203.75
South      30   165.00


In [8]:
# Named aggregation: we choose the names of the result columns
product_summary = sales.groupby("product").agg(
    total_units=("units", "sum"),
    total_revenue=("revenue", "sum"),
    avg_revenue=("revenue", "mean"),
    orders=("order_id", "count")
)
print(product_summary)

         total_units  total_revenue  avg_revenue  orders
product                                                 
Laptop            18            900        225.0       4
Phone             46            920        184.0       5
Tablet            22            330        110.0       3


In [9]:
# A custom function: the difference between the biggest and the smallest order
def revenue_range(x):
    return x.max() - x.min()

print(sales.groupby("region")["revenue"].agg(revenue_range))

# agg also works without groupby
print()
print(sales[["units", "revenue"]].agg(["sum", "mean"]).round(2))

region
East     140
North    195
South    130
Name: revenue, dtype: int64

      units  revenue
sum   86.00  2150.00
mean   7.17   179.17


#### What does this program do?
In the first cell, three functions are calculated for the revenue of each region. The output shows the sum, the mean and the maximum: East 675, 168.75 and 240, North 815, 203.75 and 300, and South 660, 165.0 and 220. The second cell uses a dictionary, so the units are added up and the revenue is averaged for each region, which gives East 30 and 168.75, North 26 and 203.75, and South 30 and 165.0.

In the third cell, named aggregation creates a summary table of the products with the names that I chose. Laptop has 18 units, a revenue of 900, an average of 225.0 and 4 orders. Phone has 46 units, 920, 184.0 and 5 orders. Tablet has 22 units, 330, 110.0 and 3 orders.

In the fourth cell, the custom function revenue_range() finds the difference between the highest and the lowest order for each region: East 140, North 195 and South 130. The last line uses agg() directly on two columns without groupby, and gives the sum and mean of units (86 and 7.17) and of revenue (2150 and 179.17).

#### AI/ML Example
agg() is used in Exploratory Data Analysis to get many statistics of the features for each class at the same time, such as the mean, the standard deviation, the minimum and the maximum of a feature for every class. It is also used for feature engineering: for each customer, the total, the average and the number of purchases are calculated, and they become new features for the model.

#### Advantages
- Many summaries in one step.
- A different function can be used for each column.
- Named aggregation gives clear column names.
- Custom functions can be used.

#### Limitations
- A custom function is slower than the built-in functions.
- The result has one row per group, so the original rows are lost.
- Without named aggregation, the column names of the result can get complicated when many functions are used.

#### Key Point
agg() gives many summaries at once. Named aggregation, like total=("revenue", "sum"), is the clearest way to write it.

## Topic 3 – transform()

### Step 1 – Understand the Concept
transform() does a group calculation but returns a result with the same number of rows as the original table. The group result is repeated for every row of that group. This is different from agg(), which gives one row per group.

Because the result lines up with the original rows, it can be added as a new column right away. transform() accepts a function name like "sum" or "mean", or a function.

### Step 2 – Demonstrate the Concept

#### Business Example
The manager wants to know what percentage of its region's total revenue each order makes up. For this, every order needs the total of its own region next to it. transform("sum") puts the region total in every row, and then the percentage can be calculated.

### Step 3 – Implement the Concept

In [10]:
# agg gives one row per group, transform gives one value for every row
print("agg shape      :", sales.groupby("region")["revenue"].sum().shape)
print("transform shape:", sales.groupby("region")["revenue"].transform("sum").shape)

# Adding the region total and the share of each order
temp = sales.copy()
temp["region_total"] = temp.groupby("region")["revenue"].transform("sum")
temp["share_pct"] = (temp["revenue"] / temp["region_total"] * 100).round(1)
print()
print(temp[["region", "revenue", "region_total", "share_pct"]].head(4))

agg shape      : (3,)
transform shape: (12,)

  region  revenue  region_total  share_pct
0  North      250           815       30.7
1  South      200           660       30.3
2   East      200           675       29.6
3  North      160           815       19.6


In [11]:
# Difference of each order from the average of its region
temp["region_avg"] = temp.groupby("region")["revenue"].transform("mean")
temp["diff_from_avg"] = temp["revenue"] - temp["region_avg"]
print(temp[["region", "revenue", "region_avg", "diff_from_avg"]].head(4))

# ML-style 1: standardising a feature inside each group
temp["z_in_region"] = temp.groupby("region")["revenue"].transform(lambda x: (x - x.mean()) / x.std())
print()
print(temp.groupby("region")["z_in_region"].mean().round(2))

  region  revenue  region_avg  diff_from_avg
0  North      250      203.75          46.25
1  South      200      165.00          35.00
2   East      200      168.75          31.25
3  North      160      203.75         -43.75

region
East     0.0
North    0.0
South    0.0
Name: z_in_region, dtype: float64


In [12]:
# ML-style 2: filling missing values with the average of the group
scores = pd.DataFrame({
    "group": ["A", "A", "B", "B", "B"],
    "score": [10, np.nan, 20, 30, np.nan]
})

scores["filled"] = scores.groupby("group")["score"].transform(lambda x: x.fillna(x.mean()))
print(scores)

  group  score  filled
0     A   10.0    10.0
1     A    NaN    10.0
2     B   20.0    20.0
3     B   30.0    30.0
4     B    NaN    25.0


#### What does this program do?
The first cell compares the shapes. agg-style groupby gives (3,), which is one value for each region, but transform gives (12,), which is one value for each of the 12 orders. Then the region total is added to every row of a copy of the table, and the share of each order is calculated. The first four orders belong to North, South, East and North, and their region totals are 815, 660, 675 and 815. Their shares are 30.7, 30.3, 29.6 and 19.6 percent.

In the second cell, the average of the region is added to every row, and the difference of each order from it is calculated. For the first four rows, the differences are 46.25, 35.0, 31.25 and -43.75. Then every revenue is standardised inside its own region with a lambda function (the value minus the region mean, divided by the region standard deviation). Because of this, the mean of the new column in each region is close to 0 in the output (it can show as 0.0 or -0.0 because of rounding).

In the third cell, a small table has missing scores in two groups. transform() with fillna(x.mean()) fills each missing score with the mean of its own group. In group A the mean is 10, so the missing value becomes 10, and in group B the mean of 20 and 30 is 25, so the missing value becomes 25. The filled column is 10, 10, 20, 30 and 25.

#### AI/ML Example
transform() is a very common feature engineering tool. It is used to fill the missing values with the average of the group instead of the average of the whole column, to create features like "the average purchase of this customer" next to every row, and to standardise or compare a value with its group. Because the result has the same number of rows, it fits straight into the training table.

#### Advantages
- Keeps all the original rows, so the result can be added as a new column.
- Makes group based features and fills in one line.
- Works with built-in function names and with custom functions.

#### Limitations
- Custom functions inside transform() are slower than the built-in ones on big data.
- The function must return a result of the same size as its group (or one value that can be repeated).
- Rows with a missing value in the group column get no result.

#### Key Point
transform() gives a group result for every original row, so the shape stays the same. It is used for group based features and for filling missing values by group.

## Topic 4 – filter()

### Step 1 – Understand the Concept
filter() works on groups. It keeps all the rows of the groups that pass a test and removes all the rows of the groups that fail. The test is a function that receives one group (a small table) and gives True or False.

This is different from normal row filtering. A normal filter checks each row on its own. filter() checks the whole group, for example "the total of the group is above 500" or "the group has at least 4 rows".

The rows that are kept are the original rows (not summarised), and they keep their original index.

### Step 2 – Demonstrate the Concept

#### Business Example
A bonus report is made only for the regions with a total revenue above 670. The total of a region is a property of the whole group, so groupby() with filter() keeps all the orders of the regions that pass and removes the orders of the other regions.

### Step 3 – Implement the Concept

In [13]:
# Keep only the regions whose total revenue is above 670
good_regions = sales.groupby("region").filter(lambda g: g["revenue"].sum() > 670)
print(good_regions)
print()
print("Shape:", good_regions.shape)
print("Regions kept:", good_regions["region"].unique().tolist())

    order_id region product quarter  units  revenue
0          1  North  Laptop      Q1      5      250
2          3   East  Laptop      Q1      4      200
3          4  North   Phone      Q1      8      160
5          6   East   Phone      Q2     12      240
6          7  North  Tablet      Q2      7      105
8          9   East  Tablet      Q2      9      135
9         10  North  Laptop      Q2      6      300
11        12   East   Phone      Q1      5      100

Shape: (8, 6)
Regions kept: ['North', 'East']


In [14]:
# ML-style: remove the rare categories (products with less than 4 orders)
common = sales.groupby("product").filter(lambda g: len(g) >= 4)
print("Shape before:", sales.shape)
print("Shape after :", common.shape)
print()
print(common["product"].value_counts())

Shape before: (12, 6)
Shape after : (9, 6)

product
Phone     5
Laptop    4
Name: count, dtype: int64


#### What does this program do?
In the first cell, the rows are grouped by region and the function checks whether the revenue total of the group is above 670. The totals are North 815, East 675 and South 660, so North and East pass and South fails. The result has the 8 rows of North and East (shape (8, 6)), and the regions kept are North and East. The rows keep their original order and their original row numbers (0, 2, 3, 5, 6, 8, 9 and 11).

In the second cell, the products are grouped and only those with at least 4 orders are kept. Laptop has 4 orders and Phone has 5, but Tablet has only 3, so all the Tablet rows are removed. The shape changes from (12, 6) to (9, 6), and the count shows Phone 5 and Laptop 4.

#### AI/ML Example
filter() is used to remove the rare categories before training. A class or a category with very few rows (like Tablet here) gives the model too little to learn from and can break the train-test split, so the groups with fewer rows than a minimum are removed. It is also used to keep only the customers or items that have enough history.

#### Advantages
- Filters on a property of a whole group, which a normal row filter cannot do.
- Keeps the original rows, not a summary.

#### Limitations
- Slower than a simple row filter, because the function is run for every group.
- The function must return a single True or False for each group.
- Removes complete groups, so a lot of data can be lost without notice.

#### Key Point
groupby().filter(function) keeps all the rows of the groups for which the function returns True, and removes all the other groups.

## Topic 5 – Pivot Tables

### Step 1 – Understand the Concept
A pivot table summarises the data in a grid, like the pivot tables of Excel. In pandas it is made with pd.pivot_table().

Main arguments:
- values : the column to calculate
- index : the column whose values become the rows
- columns : the column whose values become the column headings
- aggfunc : the function to use ("sum", "mean", "count" and so on). The default is the mean.
- fill_value : the value used for combinations that have no data (they would be NaN)
- margins=True : adds a totals row and a totals column, named All

A pivot table gives the same numbers as groupby() with two keys, but arranged as a grid that is easier to read.

### Step 2 – Demonstrate the Concept

#### Business Example
The manager wants a table with the regions in the rows and the products in the columns, and the total revenue in each cell. This is a pivot table, and it shows at a glance which region sells which product best.

### Step 3 – Implement the Concept

In [15]:
# Revenue of each region and product
pt = pd.pivot_table(sales, values="revenue", index="region", columns="product", aggfunc="sum")
print(pt)

# The average instead of the sum
print()
print(pd.pivot_table(sales, values="revenue", index="region", columns="product", aggfunc="mean"))

# With totals
print()
print(pd.pivot_table(sales, values="revenue", index="region", columns="product",
                     aggfunc="sum", margins=True))

product  Laptop  Phone  Tablet
region                        
East        200    340     135
North       550    160     105
South       150    420      90

product  Laptop  Phone  Tablet
region                        
East      200.0  170.0   135.0
North     275.0  160.0   105.0
South     150.0  210.0    90.0

product  Laptop  Phone  Tablet   All
region                              
East        200    340     135   675
North       550    160     105   815
South       150    420      90   660
All         900    920     330  2150


In [16]:
# Regions in the rows and quarters in the columns
print(pd.pivot_table(sales, values="revenue", index="region", columns="quarter", aggfunc="sum"))

# More than one value column
print()
print(pd.pivot_table(sales, values=["units", "revenue"], index="product", aggfunc="sum"))

quarter   Q1   Q2
region           
East     300  375
North    410  405
South    290  370

         revenue  units
product                
Laptop       900     18
Phone        920     46
Tablet       330     22


In [17]:
# Missing combinations: only the Q1 orders
q1 = sales[sales["quarter"] == "Q1"]

print(pd.pivot_table(q1, values="revenue", index="region", columns="product", aggfunc="sum"))

print()
print(pd.pivot_table(q1, values="revenue", index="region", columns="product",
                     aggfunc="sum", fill_value=0))

# The same numbers with groupby and unstack
print()
print(sales.groupby(["region", "product"])["revenue"].sum().unstack())

product  Laptop  Phone  Tablet
region                        
East      200.0  100.0     NaN
North     250.0  160.0     NaN
South       NaN  200.0    90.0

product  Laptop  Phone  Tablet
region                        
East        200    100       0
North       250    160       0
South         0    200      90

product  Laptop  Phone  Tablet
region                        
East        200    340     135
North       550    160     105
South       150    420      90


#### What does this program do?
In the first cell, the pivot table has the regions in the rows (East, North and South) and the products in the columns (Laptop, Phone and Tablet). The sums are East 200, 340 and 135, North 550, 160 and 105, and South 150, 420 and 90. With aggfunc="mean" the cells show the average revenue of an order: East 200, 170 and 135, North 275, 160 and 105, and South 150, 210 and 90. With margins=True a row and a column named All are added. The column totals are Laptop 900, Phone 920 and Tablet 330, the row totals are East 675, North 815 and South 660, and the grand total is 2150.

In the second cell, the quarters are used as the columns. The revenue of Q1 and Q2 is East 300 and 375, North 410 and 405, and South 290 and 370. Using values=["units", "revenue"] gives two columns for every product: Laptop 18 units and 900, Phone 46 units and 920, and Tablet 22 units and 330.

In the third cell, only the Q1 orders are used, and some region and product combinations do not exist in Q1 (for example Tablet in North), so the cells show NaN. fill_value=0 changes these NaN into 0. The last table is made with groupby() and unstack(), and it shows the same numbers as the first pivot table.

#### AI/ML Example
Pivot tables are used in data analysis to see how a target or a feature changes across two categories, for example the average purchase value for each region and each product, or the churn rate for each plan and each city. They show patterns that help to decide which features to build, and they can also be turned into features for a model.

#### Advantages
- Easy to read grid, like an Excel pivot table.
- Can use any aggregation function, and shows totals with margins.
- fill_value handles the combinations with no data.

#### Limitations
- Works well only for categories with a small number of values, otherwise the table becomes too big.
- Combinations with no data give NaN unless fill_value is used.
- Shows only summary numbers, and the detail of single rows is lost.

#### Key Point
pd.pivot_table(df, values=..., index=..., columns=..., aggfunc=...) summarises the data in a grid. Use margins=True for totals and fill_value for the empty cells.

## Topic 6 – Crosstab

### Step 1 – Understand the Concept
pd.crosstab() makes a frequency table: it counts how many rows have each combination of two (or more) categorical columns. It is like a pivot table that counts.

Useful options:
- margins=True : adds the totals.
- normalize="index" : shows the proportions of each row (every row adds up to 1). "columns" does it for the columns and True for the whole table.
- values=... and aggfunc=... : calculates another column instead of counting, like a pivot table.

The columns are given directly as Series, like pd.crosstab(df["a"], df["b"]).

### Step 2 – Demonstrate the Concept

#### Business Example
The shop wants to know how many orders each region has for each product. A crosstab of region and product gives the number of orders in every cell, without writing a calculation.

### Step 3 – Implement the Concept

In [18]:
# Number of orders for each region and product
print(pd.crosstab(sales["region"], sales["product"]))

# With totals
print()
print(pd.crosstab(sales["region"], sales["product"], margins=True))

# Proportions inside each row
print()
print(pd.crosstab(sales["region"], sales["product"], normalize="index"))

product  Laptop  Phone  Tablet
region                        
East          1      2       1
North         2      1       1
South         1      2       1

product  Laptop  Phone  Tablet  All
region                             
East          1      2       1    4
North         2      1       1    4
South         1      2       1    4
All           4      5       3   12

product  Laptop  Phone  Tablet
region                        
East       0.25   0.50    0.25
North      0.50   0.25    0.25
South      0.25   0.50    0.25


In [19]:
# Product and quarter
print(pd.crosstab(sales["product"], sales["quarter"]))

# Crosstab can also add up a column, like a pivot table
print()
print(pd.crosstab(sales["region"], sales["product"], values=sales["revenue"], aggfunc="sum"))

quarter  Q1  Q2
product        
Laptop    2   2
Phone     3   2
Tablet    1   2

product  Laptop  Phone  Tablet
region                        
East        200    340     135
North       550    160     105
South       150    420      90


#### What does this program do?
In the first cell, the crosstab counts the orders. East has 1 Laptop, 2 Phone and 1 Tablet order, North has 2, 1 and 1, and South has 1, 2 and 1. With margins=True the totals are added: each region has 4 orders, the products have 4 (Laptop), 5 (Phone) and 3 (Tablet) orders, and the grand total is 12. With normalize="index" the counts turn into proportions of each row, so for North the values are 0.50 for Laptop, 0.25 for Phone and 0.25 for Tablet, and for East and South they are 0.25, 0.50 and 0.25.

In the second cell, the crosstab of product and quarter shows Laptop 2 and 2, Phone 3 and 2, and Tablet 1 and 2 orders for Q1 and Q2. The last table uses values and aggfunc to add up the revenue instead of counting, and it gives the same numbers as the first pivot table (for example 550 for North with Laptop).

#### AI/ML Example
A crosstab is used to check the relation between two categorical columns, for example the target class and a categorical feature (like churn and contract type). If the proportions in each row are very different, the feature is likely to be useful for the model. It is also the first step of a chi-square test, and it shows if some combinations have very few rows.

#### Advantages
- Quick way to count the combinations of categories.
- normalize shows proportions directly.
- Works directly with Series, so no table name is needed.

#### Limitations
- Meant for categorical columns. A numeric column with many values gives a huge table.
- With many categories the table becomes too big to read.
- By default it only counts, and other calculations need values and aggfunc.

#### Key Point
pd.crosstab(a, b) counts the rows for every combination of two categorical columns. Use normalize for proportions and margins=True for totals.

## Interview Questions

#### 1. What is groupby() and what is the split-apply-combine idea?

groupby() splits the rows of a table into groups by the values of one or more columns, applies a calculation (like sum or mean) to each group, and combines the results into one table. For example, df.groupby("region")["revenue"].sum() gives the total revenue of each region.

#### 2. How do you calculate more than one summary for each group?

Use agg() after groupby(). It can take a list of functions like ["sum", "mean", "max"], a dictionary with a different function for each column, or named aggregation like total=("revenue", "sum"), which also gives clear names to the result columns.

#### 3. What is the difference between agg() and transform()?

agg() gives one row for each group, so the result is smaller than the original table. transform() gives a result with the same number of rows as the original table, repeating the group value for every row of the group. So transform() can be added directly as a new column, for example the region total or the group mean next to each row.

#### 4. What is the difference between pivot_table() and crosstab()?

Both make a grid with one column's values as the rows and another column's values as the headings. pivot_table() calculates a chosen column with a function like sum or mean (using values and aggfunc). crosstab() by default counts the number of rows for each combination of categories, and it is used for frequency tables. crosstab() can also calculate a column if values and aggfunc are given.

#### 5. What does filter() do on a groupby object and how is it different from a normal row filter?

groupby().filter(function) keeps all the rows of the groups for which the function returns True and removes all the rows of the other groups, for example keeping only the products that have at least 4 orders. A normal row filter checks every row on its own, but filter() decides for a whole group based on a property of the group, like its size or its total.